# Platinum 2 performance (val only)

In [1]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)

HERE = Path.cwd().resolve()
REPO = None
for p in [HERE, *HERE.parents]:
    if (p / "platinum2" / "results" / "leaderboard.csv").exists() or (p / "src" / "gold").exists():
        REPO = p
        break
if REPO is None:
    raise FileNotFoundError("repo root not found")

RESULTS = REPO / "platinum2" / "results"
FIG = RESULTS / "figures"
FIG.mkdir(parents=True, exist_ok=True)
GOLD = REPO / "data" / "gold" / "delay" / "platinum2"

lb = pd.read_csv(RESULTS / "leaderboard.csv") if (RESULTS / "leaderboard.csv").exists() else pd.DataFrame()
audit = {}
if (GOLD / "series_audit.json").exists():
    audit = json.loads((GOLD / "series_audit.json").read_text(encoding="utf-8"))
display(Markdown("# Platinum 2 — delayed MW sitting in GIA next year"))
display(Markdown(
    "Not project-row COD months. This is the **stock of delayed megawatts** in the EIA planned inventory. "
    "GIA is a thinner matched overlay. 2024 is sealed."
))
display(Markdown(
    f"Snapshots **{audit.get('n_snapshots')}**. "
    f"Enough for a real series: **{audit.get('enough_for_timesfm')}**."
))
display(lb)

# Platinum 2 — delayed MW sitting in GIA next year

Not project-row COD months. This is the **stock of delayed megawatts** in the EIA planned inventory. GIA is a thinner matched overlay. 2024 is sealed.

Snapshots **18**. Enough for a real series: **True**.

,model,split,status,rmse_h12,mape_h12,rmse_h3,mape_h3,n_folds_h12,n_folds_h3,beats_seasonal_naive,reason
0,timesfm,val,ok,10714.657132,0.587370,3358.288737,0.165551,4.0,3.0,True,NaN
1,holt,val,ok,12684.455454,0.721716,5140.250467,0.249321,4.0,3.0,True,NaN
2,seasonal_naive,val,ok,14864.238911,0.815436,15646.660918,0.753915,4.0,3.0,False,NaN
3,naive,val,ok,14864.238911,0.815436,5272.210899,0.203436,4.0,3.0,False,NaN
4,ridge_lags,val,ok,14864.238911,0.815436,14333.483047,0.490909,4.0,3.0,False,NaN
5,ma3,val,ok,16559.367425,0.891047,8501.037178,0.404096,4.0,3.0,False,NaN
6,arima,val,skipped,NaN,NaN,NaN,NaN,4.0,3.0,False,ARIMA did not identify / all-NaN forecasts


## Series

In [2]:
series_path = GOLD / "gia_delayed_mw_monthly.parquet"
if series_path.exists():
    s = pd.read_parquet(series_path)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=s["year_month"], y=s["delayed_mw"], mode="lines+markers", name="delayed MW (EIA planned)"))
    if "gia_delayed_mw" in s.columns:
        fig.add_trace(go.Scatter(x=s["year_month"], y=s["gia_delayed_mw"], mode="lines+markers", name="GIA-matched delayed MW"))
    fig.update_layout(title="Delayed planned MW over EIA quarters", yaxis_title="MW")
    fig.write_html(FIG / "series.html")
    try:
        fig.show()
    except Exception as e:
        print("show skipped:", e)
else:
    display(Markdown("_series missing — run scripts/run_platinum2_gold.py_"))

## Leaderboard

In [3]:
ok = lb[lb["status"].eq("ok")].copy() if len(lb) else lb
if len(ok) and "rmse_h12" in ok.columns:
    fig = go.Figure(go.Bar(x=ok["model"], y=ok["rmse_h12"], name="RMSE h=12m"))
    fig.update_layout(title="12-month-ahead RMSE (MW). Lower is better. Test sealed.")
    fig.write_html(FIG / "rmse_h12.html")
    try:
        fig.show()
    except Exception as e:
        print("show skipped:", e)
if len(ok) and "rmse_h3" in ok.columns:
    fig = go.Figure(go.Bar(x=ok["model"], y=ok["rmse_h3"], name="RMSE h=3m"))
    fig.update_layout(title="3-month-ahead RMSE (sanity)")
    fig.write_html(FIG / "rmse_h3.html")
    try:
        fig.show()
    except Exception as e:
        print("show skipped:", e)

In [4]:
display(Markdown("## What this is not"))
display(Markdown(
    "- Not a per-project COD calendar (that was Platinum 1, and it did not work).\\n"
    "- Not a MISO Firm Service Step-Up failure record.\\n"
    "- EIA planned-COD slip in MISO-footprint states, with a GIA match overlay."
))

## What this is not

- Not a per-project COD calendar (that was Platinum 1, and it did not work).\n- Not a MISO Firm Service Step-Up failure record.\n- EIA planned-COD slip in MISO-footprint states, with a GIA match overlay.